# HW2 final pipeline: test-time augmentation, fusion and temporal rescoring

Model: **YOLO11s trained at 1280 px** (`yolo_1280`, best.pt from the main notebook). No training happens here.

| Part | What | Needs |
|---|---|---|
| 1 | data tables (all val frames, test frames), COCO ground truth | competition data |
| 2 | **GPU**: raw detections of every pass (scale x flip) on all val and test frames, saved as `raw_*.pkl` | GPU, `best.pt` |
| 3 | **CPU**: fusion of the passes (WBF / NMS, IoU, 300 or 500 detections), selected on the val frames | `raw_*.pkl` |
| 4 | **CPU**: temporal rescoring of detections along the video (bonus *Data Temporality*), selected on all val frames | `raw_*.pkl` |
| 5 | submission files and the check of their format | |

**How to run (saves GPU quota)**
1. Session with GPU: `MODE = "infer"`. Input: competition data + the output of the `yolo_1280` run (it contains `runs/yolo_1280/weights/best.pt`). About 25-40 min.
2. Session without GPU (Accelerator: None): `MODE = "post"`. Input: competition data + the output of step 1 (it contains `raw_*.pkl`). About 40-60 min on CPU.
3. `MODE = "all"` does both in one GPU session (more GPU time is used by the CPU part).
4. Before a real run you can set `SMOKE = True`: a few frames, 5 minutes, everything is executed (files get the prefix `smoke_`).

Selection rules: fusion settings are chosen by `mAP_testlike` on the val frames kept in the main notebook (every 4th frame); temporal settings on all consecutive val frames. "testlike" means frames up to 1920 px wide, like the test set.

In [1]:
from pathlib import Path
import os, sys, json, time, pickle, io, contextlib, subprocess, shutil, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ---------------- what to do ----------------
MODE = "infer"        # "infer": GPU, raw detections of every pass -> .pkl | "post": CPU, fusion + temporal + submissions | "all": both
SMOKE = False       # True: a few frames only, checks that everything runs (files get the prefix smoke_)
WEIGHTS = None      # path to runs/yolo_1280/weights/best.pt; None = search /kaggle/input
RAW_DIR = None      # folder with raw_*.pkl for MODE = "post"; None = search /kaggle/working and /kaggle/input

# ---------------- test-time augmentation ----------------
SCALES = [1280, 1600]    # long side of the network input in px (the model was trained at 1280)
FLIPS = [False, True]    # horizontal flip
CONF, MAX_DET_PASS = 0.001, 300
CAPS = [300, 500]        # detections kept per frame after fusion (the competition scores at most 500)
SEED = 42

ROOT = Path(os.environ.get("HW2_ROOT", "/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026"))
if not ROOT.exists():
    ROOT = Path("/kaggle/input/multiple-object-video-detection-hw-2-ucu-ai-2026")
WORK = Path(os.environ.get("HW2_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
TAG = "smoke_" if SMOKE else ""
assert MODE in ("infer", "post", "all")
np.random.seed(SEED)
print("final pipeline notebook | MODE:", MODE, "| smoke:", SMOKE, "| ROOT:", ROOT)

final pipeline notebook | MODE: infer | smoke: False | ROOT: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026


In [2]:
!pip install -q ultralytics     # needs Internet = On

## 1. Data tables
All **2846 val frames** are used here (not only every 4th frame), because the temporal part needs consecutive frames, exactly like in the test videos. The frames that the main notebook used for validation (every 4th frame of each video) are marked as `SUB4`, so that numbers stay comparable with the earlier tables.

In [3]:
VAL_JSON = ROOT / "val" / "annotations_coco.json"
vi = pd.DataFrame(json.loads(VAL_JSON.read_text())["images"]).sort_values(["video_id", "frame_id"]).reset_index(drop=True)
vi["rank"] = vi.groupby("video_id").cumcount()                   # position of the frame inside its video
val_frames = pd.DataFrame({"key": vi["id"].astype(int), "split": "val", "path": [str(ROOT / "val" / f) for f in vi.file_name],
                           "clip": vi.video_id.astype(str), "order": vi.frame_id.astype(int), "rank": vi["rank"],
                           "width": vi.width.astype(int), "height": vi.height.astype(int)})
t = pd.read_csv(ROOT / "test_images.csv").sort_values(["clip", "frame"]).reset_index(drop=True)
test_frames = pd.DataFrame({"key": t.image_id, "split": "test", "path": [str(ROOT / f) for f in t.file_name], "clip": t["clip"],
                            "order": t.frame.astype(int), "rank": t.groupby("clip").cumcount(),
                            "width": t.width.astype(int), "height": t.height.astype(int)})
if SMOKE:   # a few frames of two videos
    val_frames = val_frames[val_frames["clip"].isin(val_frames["clip"].unique()[:2]) & (val_frames["rank"] < 10)].reset_index(drop=True)
    test_frames = test_frames[test_frames["clip"].isin(test_frames["clip"].unique()[:2]) & (test_frames["rank"] < 6)].reset_index(drop=True)

SUB4 = set(val_frames[val_frames["rank"] % 4 == 0].key)          # the validation frames of the main notebook
TESTLIKE = set(val_frames[val_frames.width <= 1920].key)         # test frames are at most 1920 px wide
CLIPS = {name: [g.key.tolist() for _, g in df.groupby("clip", sort=False)]
         for name, df in (("val", val_frames), ("test", test_frames))}   # frame keys of every video in time order

for name, df in (("val", val_frames), ("test", test_frames)):
    d = df.groupby("clip").order.diff().dropna()
    print(f"{name}: {len(df)} frames, {df["clip"].nunique()} videos | consecutive neighbours: {(d == 1).mean():.1%}")
print("val frames kept for fusion selection (every 4th):", len(SUB4), "| test-like (<=1920 px):", len(TESTLIKE))

val: 2846 frames, 7 videos | consecutive neighbours: 100.0%
test: 3811 frames, 48 videos | consecutive neighbours: 100.0%
val frames kept for fusion selection (every 4th): 714 | test-like (<=1920 px): 1286


## 2. Raw detections of every pass (GPU)
Each frame is read once. For every pass (scale, flip) the frame is shrunk by us to the scale (long side, `INTER_AREA`, never enlarged), optionally flipped, and passed to the network with a low confidence threshold. Boxes are flipped back and scaled to the original pixel coordinates. The result of each pass is saved separately, so that fusion can be tried again without a GPU.

In [4]:
PASSES = [(s, f) for s in SCALES for f in FLIPS]

def find_weights():
    if WEIGHTS:
        return Path(WEIGHTS)
    hits = []
    for dp, dn, fn in os.walk("/kaggle/input"):
        if "multiple-object-video-detection" in dp or dp.count(os.sep) - 2 > 7:
            dn[:] = []
            continue
        if "best.pt" in fn:
            hits.append(Path(dp) / "best.pt")
    pref = [h for h in hits if "yolo_1280" in str(h)]
    if len(pref) == 1 or (not pref and len(hits) == 1):
        return (pref or hits)[0]
    raise FileNotFoundError(f"best.pt of yolo_1280 not found or ambiguous: {hits}. "
                            "Add the output of the yolo_1280 run as Input or set WEIGHTS = '...'.")

if MODE in ("infer", "all"):
    import torch, cv2
    from ultralytics import YOLO
    assert torch.cuda.is_available(), "MODE = 'infer' needs the GPU"
    W_PATH = find_weights()
    print("weights:", W_PATH)
    model = YOLO(str(W_PATH))

    def predict_pass(img, scale, flip):
        h0, w0 = img.shape[:2]
        r = min(1.0, scale / max(h0, w0))                              # only shrink, never enlarge
        x = cv2.resize(img, (round(w0 * r), round(h0 * r)), interpolation=cv2.INTER_AREA) if r < 1.0 else img
        if flip:
            x = np.ascontiguousarray(x[:, ::-1])
        res = model.predict(x, imgsz=scale, conf=CONF, max_det=MAX_DET_PASS, device=0, verbose=False)[0]
        b = res.boxes.xyxy.cpu().numpy().astype(np.float32)
        if flip and len(b):
            b = np.stack([x.shape[1] - b[:, 2], b[:, 1], x.shape[1] - b[:, 0], b[:, 3]], 1)   # undo the flip
        return b / r, res.boxes.conf.cpu().numpy().astype(np.float32), res.boxes.cls.cpu().numpy().astype(np.int16)

    def run_inference(frames, split):
        out = {p: {} for p in PASSES}
        first = cv2.imread(frames.path.iloc[0])
        for p in PASSES:
            predict_pass(first, *p)                                    # warm-up, not timed
        size_mismatch, t0 = 0, time.time()
        for r in tqdm(list(frames.itertuples()), desc=split):
            img = cv2.imread(r.path)
            assert img is not None, r.path
            size_mismatch += img.shape[:2] != (r.height, r.width)
            for p in PASSES:
                out[p][r.key] = predict_pass(img, *p)
        print(f"{split}: {(time.time() - t0) / len(frames) * 1000:.0f} ms per frame for {len(PASSES)} passes | size differs from the table: {size_mismatch}")
        for (s, f), d in out.items():
            with open(WORK / f"{TAG}raw_{split}_{s}_{int(f)}.pkl", "wb") as fh:
                pickle.dump(d, fh)

    run_inference(val_frames, "val")
    run_inference(test_frames, "test")
    print("raw detections saved to", WORK)
    if MODE == "infer":
        print("Next: a session without GPU with MODE = 'post' and this output as Input.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
weights: /kaggle/input/notebooks/bohdanazubrytska/hw2-one-notebook/runs/yolo_1280/weights/best.pt


val:   0%|          | 0/2846 [00:00<?, ?it/s]

val: 280 ms per frame for 4 passes | size differs from the table: 0


test:   0%|          | 0/3811 [00:00<?, ?it/s]

test: 217 ms per frame for 4 passes | size differs from the table: 0
raw detections saved to /kaggle/working
Next: a session without GPU with MODE = 'post' and this output as Input.


## 3. Fusion of the passes (CPU)
**Weighted Boxes Fusion** (Solovyev et al., 2021, `ensemble-boxes` package): boxes of the same class from different passes with IoU above a threshold form a cluster; the fused box is the score-weighted average of the cluster, its score is the mean score multiplied by (found in n passes / all passes), so boxes seen in few passes are down-weighted. **NMS** is the alternative that keeps only the best box of a cluster.
The selection is done on the validation frames of the main notebook (every 4th frame, 714 frames), the COCO evaluation uses `maxDets` equal to the cap, because the competition scores up to 500 detections per frame.

In [5]:
if MODE in ("post", "all"):
    try:
        import ensemble_boxes as eb
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ensemble-boxes"], check=True)
        import ensemble_boxes as eb
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    def find_file(name):
        cands = ([Path(RAW_DIR)] if RAW_DIR else []) + [WORK]
        for p in cands:
            if (p / name).exists():
                return p / name
        for dp, dn, fn in os.walk("/kaggle/input"):
            if "multiple-object-video-detection" in dp or dp.count(os.sep) - 2 > 6:
                dn[:] = []
                continue
            if name in fn:
                return Path(dp) / name
        raise FileNotFoundError(f"{name} not found: run MODE = 'infer' first or set RAW_DIR")

    def load_raw(split):
        out = {}
        for s, f in PASSES:
            with open(find_file(f"{TAG}raw_{split}_{s}_{int(f)}.pkl"), "rb") as fh:
                out[(s, f)] = pickle.load(fh)
        return out

    RAW = {"val": load_raw("val"), "test": load_raw("test")}
    print("raw detections loaded:", {k: len(v) for k, v in RAW.items()}, "passes each")

    with contextlib.redirect_stdout(io.StringIO()):
        GT = COCO(str(VAL_JSON))                  # iscrowd = 1 are the ignored regions

    CLASS_NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]

    def to_coco(dets, cap):
        out = []
        for k, (b, s, c) in dets.items():
            for (x1, y1, x2, y2), sc, cl in zip(b[:cap], s[:cap], c[:cap]):
                out.append({"image_id": int(k), "category_id": int(cl) + 1, "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(sc)})
        return out

    def _ap(E, iou=None, area=0):
        p = E.eval["precision"]                   # [IoU, recall, class, area, maxDets]
        if iou is not None:
            p = p[np.isclose(E.params.iouThrs, iou)]
        p = p[:, :, :, area, -1]
        p = p[p > -1]
        return float(p.mean()) if p.size else 0.0

    def coco_eval(dets, ids, cap):
        ids = list(ids)
        cd = to_coco({k: dets[k] for k in ids}, cap)
        if not cd:
            return {"mAP": 0.0, "AP50": 0.0, "AP_small": 0.0, "AP_large": 0.0}
        with contextlib.redirect_stdout(io.StringIO()):
            E = COCOeval(GT, GT.loadRes(cd), "bbox")
            E.params.imgIds = ids
            E.params.maxDets = [1, 10, cap]
            E.evaluate(); E.accumulate()
        return {"mAP": _ap(E), "AP50": _ap(E, 0.5), "AP_small": _ap(E, area=1), "AP_large": _ap(E, area=3)}

    def score(dets, ids, cap):
        """All given frames and the test-like frames (<= 1920 px) of them."""
        a = coco_eval(dets, ids, cap)
        tl = coco_eval(dets, [k for k in ids if k in TESTLIKE], cap)
        return {**a, "mAP_testlike": tl["mAP"], "AP_small_testlike": tl["AP_small"]}

    def fuse_frame(lst, W, H, how, iou, cap):
        if len(lst) == 1:                                           # single pass: only sort and cut
            b, s, c = lst[0]
            o = np.argsort(-s)[:cap]
            return b[o], s[o], c[o]
        wh = np.array([W, H, W, H], np.float32)
        bl, sl, ll = [], [], []
        for b, s, c in lst:
            bn = np.clip(b / wh, 0, 1) if len(b) else np.zeros((0, 4), np.float32)
            ok = ((bn[:, 2] - bn[:, 0]) > 1e-6) & ((bn[:, 3] - bn[:, 1]) > 1e-6) if len(bn) else np.zeros(0, bool)
            bl.append(bn[ok]); sl.append(s[ok]); ll.append(c[ok].astype(int))
        if how == "wbf":
            fb, fs, fl = eb.weighted_boxes_fusion(bl, sl, ll, iou_thr=iou, skip_box_thr=CONF, conf_type="avg")
        else:
            fb, fs, fl = eb.nms(bl, sl, ll, iou_thr=iou)
        o = np.argsort(-fs)[:cap]
        return (fb[o] * wh).astype(np.float32), fs[o].astype(np.float32), fl[o].astype(np.int16)

    def fuse_all(frames, split, passes, how, iou, cap, desc=""):
        raw = RAW[split]
        return {r.key: fuse_frame([raw[p][r.key] for p in passes], r.width, r.height, how, iou, cap)
                for r in tqdm(list(frames.itertuples()), desc=desc, leave=False)}

In [6]:
if MODE in ("post", "all"):
    F_SUB = val_frames[val_frames.key.isin(SUB4)]
    IDS_SUB = list(F_SUB.key)
    CAP_MAX = max(CAPS)
    S0 = SCALES[0]
    configs = [(f"single {s}", "-", np.nan, [(s, False)]) for s in SCALES]
    configs += [(f"flip {S0}", "wbf", 0.6, [(S0, False), (S0, True)]),
                (f"scales {'+'.join(map(str, SCALES))}", "wbf", 0.6, [(s, False) for s in SCALES])]
    configs += [(f"scales {'+'.join(map(str, SCALES))} + flips", "wbf", i, PASSES) for i in (0.6, 0.7, 0.8)]
    configs += [(f"scales {'+'.join(map(str, SCALES))} + flips", "nms", i, PASSES) for i in (0.6, 0.7)]

    rows = []
    for mode, how, iou, passes in configs:
        dets = fuse_all(F_SUB, "val", passes, how if how != "-" else "wbf", iou if how != "-" else 0.6, CAP_MAX, desc=f"{mode} {how} {iou}")
        for cap in CAPS:
            rows.append({"mode": mode, "how": how, "iou": iou, "passes": len(passes), "cap": cap, **score(dets, IDS_SUB, cap)})
        print(pd.DataFrame(rows[-len(CAPS):]).round(4).to_string(header=False, index=False))
    FUSION = pd.DataFrame(rows).sort_values("mAP_testlike", ascending=False).reset_index(drop=True)
    FUSION.to_csv(WORK / f"{TAG}inference_experiments.csv", index=False)
    print("\nFusion experiments (val, every 4th frame), best first:")
    print(FUSION.round(4).to_string(index=False))
    BEST = FUSION.iloc[0]
    BEST_PASSES = next(ps for m, _, _, ps in configs if m == BEST["mode"])
    print("\nselected:", dict(BEST[["mode", "how", "iou", "cap"]]))

## 4. Temporal rescoring (bonus: Data Temporality)
Consecutive frames of a drone video show the same objects at almost the same place, so a real object is detected in neighbouring frames, while a false positive usually flickers. For every detection of frame *t* we look for a detection of the **same class** with IoU >= `thr` in the neighbouring frames and mix the score:

`score_new = (1 - alpha) * score + alpha * mean(score of the best match in every neighbour frame, 0 if there is none)`

Variants: only the previous frame (causal), previous + next frame, two frames on each side. **Propagation** additionally copies confident detections of the previous frame that have no match in the current frame (a missed object), with a decayed score. Everything is evaluated on all consecutive val frames and applied to the test videos in the same way. The simple linking by IoU is in the spirit of Seq-NMS / T-CNN (see the reading list); camera motion is not compensated, which is acceptable because the boxes move little between consecutive frames.

In [7]:
if MODE in ("post", "all"):
    def iou_matrix(a, b):
        ix = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
        iy = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
        inter = ix * iy
        aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
        ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
        return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

    def match(cur, other, thr):
        """For every detection of cur: score of the best same-class detection of other with IoU >= thr (else 0), and the IoU matrix."""
        (b, s, c), (bo, so, co) = cur, other
        if len(b) == 0 or len(bo) == 0:
            return np.zeros(len(b), np.float32), np.zeros((len(b), len(bo)), np.float32)
        M = iou_matrix(b, bo)
        M[c[:, None] != co[None, :]] = 0
        j = M.argmax(1)
        best = M[np.arange(len(b)), j]
        return np.where(best >= thr, so[j], 0.0).astype(np.float32), M

    def temporal_rescore(clips, dets, alpha, thr, offsets, prop=None, cap=500):
        out = {}
        for keys in clips:
            for t, k in enumerate(keys):
                b, s, c = dets[k]
                s_new = s.copy()
                nb = [t + o for o in offsets if 0 <= t + o < len(keys)]
                if alpha > 0 and nb and len(b):
                    acc = np.zeros(len(b), np.float32)
                    for u in nb:
                        acc += match((b, s, c), dets[keys[u]], thr)[0]
                    s_new = ((1 - alpha) * s + alpha * acc / len(nb)).astype(np.float32)
                if prop and t > 0:                                      # propagate confident detections of the previous frame
                    tau, decay = prop
                    bp, sp, cp = dets[keys[t - 1]]
                    sel = sp >= tau
                    if sel.any():
                        bp, sp, cp = bp[sel], sp[sel], cp[sel]
                        if len(b):
                            _, M = match((bp, sp, cp), (b, s, c), thr)            # previous detections vs current ones
                            new = M.max(1) < thr if M.size else np.ones(len(bp), bool)
                        else:
                            new = np.ones(len(bp), bool)
                        if new.any():
                            b = np.concatenate([b, bp[new]]); c = np.concatenate([c, cp[new]])
                            s_new = np.concatenate([s_new, sp[new] * decay]).astype(np.float32)
                o = np.argsort(-s_new)[:cap]
                out[k] = (b[o], s_new[o], c[o])
        return out

    CAP_FINAL = int(BEST["cap"])
    DETS_VAL = fuse_all(val_frames, "val", BEST_PASSES, BEST["how"] if BEST["how"] != "-" else "wbf", BEST["iou"] if BEST["how"] != "-" else 0.6,
                        CAP_FINAL, desc="fuse all val frames")
    IDS_VAL = list(val_frames.key)
    base = {"variant": "no temporal step", "alpha": 0.0, "offsets": "-", "propagate": "-", **score(DETS_VAL, IDS_VAL, CAP_FINAL)}
    trows = [base]
    print(pd.DataFrame([base]).round(4).to_string(index=False))

    TEMP_GRID = [("previous frame", (-1,), 0.25, None), ("previous frame", (-1,), 0.5, None),
                 ("previous + next frame", (-1, 1), 0.25, None), ("previous + next frame", (-1, 1), 0.5, None),
                 ("2 previous + 2 next", (-2, -1, 1, 2), 0.25, None), ("2 previous + 2 next", (-2, -1, 1, 2), 0.5, None),
                 ("previous frame + propagate", (-1,), 0.0, (0.4, 0.5)),
                 ("previous + next frame + propagate", (-1, 1), 0.5, (0.4, 0.5))]
    THR = 0.5
    for name, offs, alpha, prop in tqdm(TEMP_GRID, desc="temporal variants"):
        d = temporal_rescore(CLIPS["val"], DETS_VAL, alpha, THR, offs, prop, CAP_FINAL)
        trows.append({"variant": name, "alpha": alpha, "offsets": str(offs), "propagate": str(prop) if prop else "-", **score(d, IDS_VAL, CAP_FINAL)})
        print(pd.DataFrame(trows[-1:]).round(4).to_string(header=False, index=False))
    TEMP = pd.DataFrame(trows)
    TEMP["gain_mAP_testlike"] = TEMP["mAP_testlike"] - base["mAP_testlike"]
    TEMP.to_csv(WORK / f"{TAG}temporal_experiments.csv", index=False)
    print("\nTemporal experiments (all consecutive val frames):")
    print(TEMP.round(4).to_string(index=False))

## 5. Submissions
Format (competition page): one row per test frame, `image_id, PredictionString`, groups `class score x y w h` (class 0..7, x/y top-left corner in pixels of the original frame), at most 500 groups per frame, an empty string for a frame without detections.
The temporal step is used for the final file only if it improves `mAP_testlike` on the val videos by at least 0.002.

In [8]:
if MODE in ("post", "all"):
    HEADER, MAX_PER_IMAGE = ["image_id", "PredictionString"], 500

    def make_pred_string(b, s, c, W, H):
        b, s, c = np.asarray(b, float).reshape(-1, 4), np.asarray(s, float), np.asarray(c)
        parts = []
        for i in np.argsort(-s)[:MAX_PER_IMAGE]:
            x1, y1, x2, y2 = max(0.0, b[i, 0]), max(0.0, b[i, 1]), min(float(W), b[i, 2]), min(float(H), b[i, 3])
            w, h = x2 - x1, y2 - y1
            if round(w, 1) <= 0 or round(h, 1) <= 0:
                continue
            parts.append(f"{int(c[i])} {s[i]:.5f} {x1:.1f} {y1:.1f} {w:.1f} {h:.1f}")
        return " ".join(parts)

    def write_submission(dets, path):
        rows = [(r.key, make_pred_string(*dets[r.key], r.width, r.height)) for r in test_frames.itertuples()]
        pd.DataFrame(rows, columns=HEADER).to_csv(path, index=False)

    def check_submission(path):
        sub = pd.read_csv(path, keep_default_na=False, dtype=str)
        assert list(sub.columns) == HEADER and len(sub) == len(test_frames)
        assert sub.image_id.is_unique and set(sub.image_id) == set(test_frames.key), "image_id do not match test_images.csv"
        toks = sub.PredictionString.str.split()
        n = toks.map(len)
        assert (n % 6 == 0).all() and (n // 6).max() <= MAX_PER_IMAGE
        a = np.array([x for tk in toks for x in tk], dtype=float).reshape(-1, 6)
        assert set(a[:, 0].astype(int)) <= set(range(8)) and (a[:, 1] >= 0).all() and (a[:, 1] <= 1).all() and (a[:, 4:] > 0).all()
        print(f"{Path(path).name}: OK, {len(sub)} rows, {len(a)} detections, mean {len(a) / len(sub):.0f} per frame, max {(n // 6).max()}")

    # test frames: fusion with the selected setting
    DETS_TEST = fuse_all(test_frames, "test", BEST_PASSES, BEST["how"] if BEST["how"] != "-" else "wbf", BEST["iou"] if BEST["how"] != "-" else 0.6,
                         CAP_FINAL, desc="fuse test frames")
    write_submission(DETS_TEST, WORK / f"{TAG}submission_fusion.csv"); check_submission(WORK / f"{TAG}submission_fusion.csv")

    cand = TEMP.iloc[1:].sort_values("mAP_testlike", ascending=False).iloc[0]
    use_temporal = cand["gain_mAP_testlike"] >= 0.002
    final_dets = DETS_TEST
    if use_temporal:
        import ast
        offs = ast.literal_eval(cand["offsets"]); prop = ast.literal_eval(cand["propagate"]) if cand["propagate"] != "-" else None
        final_dets = temporal_rescore(CLIPS["test"], DETS_TEST, float(cand["alpha"]), THR, offs, prop, CAP_FINAL)
    write_submission(final_dets, WORK / f"{TAG}submission_final.csv"); check_submission(WORK / f"{TAG}submission_final.csv")
    shutil.copy2(WORK / f"{TAG}submission_final.csv", WORK / f"{TAG}submission.csv")

    final_cfg = {"fusion": {k: (None if pd.isna(v) else (v.item() if hasattr(v, "item") else v)) for k, v in BEST[["mode", "how", "iou", "cap", "mAP", "mAP_testlike"]].items()},
                 "temporal_used": bool(use_temporal),
                 "temporal": ({k: (v.item() if hasattr(v, "item") else v) for k, v in cand[["variant", "alpha", "offsets", "propagate", "mAP_testlike", "gain_mAP_testlike"]].items()} if use_temporal else None),
                 "weights": str(W_PATH) if MODE == "all" else "see the infer session", "scales": SCALES, "flips": FLIPS, "conf": CONF, "seed": SEED}
    (WORK / f"{TAG}final_config.json").write_text(json.dumps(final_cfg, indent=1, default=str))
    print(json.dumps(final_cfg, indent=1, default=str))

In [9]:
if MODE in ("post", "all"):
    # figure for the report: test-like mAP of all fusion settings and of the temporal variants
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    f = FUSION[FUSION.cap == CAP_FINAL].iloc[::-1]
    ax[0].barh([f"{m} {h} {i}" if h != "-" else m for m, h, i in zip(f["mode"], f["how"], f["iou"])], f["mAP_testlike"], color="tab:blue")
    ax[0].set_title(f"Fusion (val every 4th frame, cap {CAP_FINAL})"); ax[0].set_xlabel("mAP, frames <= 1920 px")
    ax[1].barh(TEMP["variant"] + " a=" + TEMP["alpha"].astype(str), TEMP["mAP_testlike"], color="tab:green")
    ax[1].axvline(base["mAP_testlike"], color="k", ls="--", lw=1); ax[1].set_title("Temporal rescoring (all val frames)"); ax[1].set_xlabel("mAP, frames <= 1920 px")
    plt.tight_layout(); plt.savefig(WORK / f"{TAG}final_pipeline.png", dpi=130); plt.show()
    with zipfile.ZipFile(WORK / f"{TAG}results_final_pipeline.zip", "w", zipfile.ZIP_DEFLATED) as z:    # small files only
        for fn in ("inference_experiments.csv", "temporal_experiments.csv", "final_config.json", "final_pipeline.png"):
            z.write(WORK / f"{TAG}{fn}", fn)
    print("results_final_pipeline.zip written")